In [25]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("ntphiep/viT5_tst_coarse", use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained("ntphiep/viT5_tst_coarse")

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'MT5Tokenizer'. 
The class this function is called from is 'T5Tokenizer'.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [4]:
import sagemaker
import boto3
from sagemaker.huggingface import HuggingFaceModel

role = "arn:aws:iam::014498663963:role/service-role/AmazonSageMaker-ExecutionRole-20250504T220510"

# Hub Model configuration. https://huggingface.co/models
hub = {
	'HF_MODEL_ID':'ntphiep/viT5_tst_coarse',
	'HF_TASK':'text-generation'
}

# create Hugging Face Model Class
huggingface_model = HuggingFaceModel(
	transformers_version='4.49.0',
	pytorch_version='2.6.0',
	py_version='py312',
	env=hub,
	role=role, 
)

# deploy model to SageMaker Inference
predictor = huggingface_model.deploy(
	initial_instance_count=1, # number of instances
	instance_type='ml.m4.xlarge' # ec2 instance type
)

predictor.predict({
	"inputs": "Can you please let us know more details about your ",
})

------------------------------------------------*

Please check the troubleshooting guide for common errors: https://docs.aws.amazon.com/sagemaker/latest/dg/sagemaker-python-sdk-troubleshooting.html#sagemaker-python-sdk-troubleshooting-create-endpoint


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:23                                                                                   │
│                                                                                                  │
│   20 )                                                                                           │
│   21                                                                                             │
│   22 # deploy model to SageMaker Inference                                                       │
│ ❱ 23 predictor = huggingface_model.deploy(                                                       │
│   24 │   initial_instance_count=1, # number of instances                                         │
│   25 │   instance_type='ml.m4.xlarge' # ec2 instance type                                        │
│   26 )                                                                                           │
│                                                                                                  │
│ C:\Users\Hiep\AppData\Roaming\Python\Python312\site-packages\sagemaker\huggingface\model.py:326  │
│ in deploy                                                                                        │
│                                                                                                  │
│   323 │   │   │   │   inference_tool=inference_tool,                                             │
│   324 │   │   │   )                                                                              │
│   325 │   │                                                                                      │
│ ❱ 326 │   │   return super(HuggingFaceModel, self).deploy(                                       │
│   327 │   │   │   initial_instance_count,                                                        │
│   328 │   │   │   instance_type,                                                                 │
│   329 │   │   │   serializer,                                                                    │
│                                                                                                  │
│ C:\Users\Hiep\AppData\Roaming\Python\Python312\site-packages\sagemaker\model.py:1814 in deploy   │
│                                                                                                  │
│   1811 │   │   │   │   )                                                                         │
│   1812 │   │   │   │   self.sagemaker_session.update_endpoint(self.endpoint_name, endpoint_conf  │
│   1813 │   │   │   else:                                                                         │
│ ❱ 1814 │   │   │   │   self.sagemaker_session.endpoint_from_production_variants(                 │
│   1815 │   │   │   │   │   name=self.endpoint_name,                                              │
│   1816 │   │   │   │   │   production_variants=[production_variant],                             │
│   1817 │   │   │   │   │   tags=tags,                                                            │
│                                                                                                  │
│ C:\Users\Hiep\AppData\Roaming\Python\Python312\site-packages\sagemaker\session.py:6033 in        │
│ endpoint_from_production_variants                                                                │
│                                                                                                  │
│   6030 │   │   logger.info("Creating endpoint-config with name %s", name)                        │
│   6031 │   │   self.sagemaker_client.create_endpoint_config(**config_options)                    │
│   6032 │   │                                                                                     │
│ ❱ 6033 │   │   return self.create_endpoint(                                                      │
│   6034 │   │   │   endpoint_name=name,                     

In [13]:
from transformers import MT5Tokenizer, AutoModelForSeq2SeqLM

tokenizer = MT5Tokenizer.from_pretrained("ntphiep/viT5_tst_formal")
model = AutoModelForSeq2SeqLM.from_pretrained("ntphiep/viT5_tst_formal")

def predict(text):
    inputs = tokenizer(text, return_tensors="pt", padding='longest', max_length=64)
    input_ids = inputs.input_ids
    attention_mask = inputs.attention_mask
    output = model.generate(input_ids, attention_mask=attention_mask, max_length=256, top_p=0.95)
    return tokenizer.decode(output[0], skip_special_tokens=True)


text = "Bọn công nhân thì được trả bằng thóc, một thằng bình thường kiếm được có 5 bao rưỡi thóc một tháng, còn thằng quản đốc thì được tận 7 bao rưỡi."
result = predict(text) 
print("👉 Output:", result)


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'MT5Tokenizer'. 
The class this function is called from is 'T5Tokenizer'.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


👉 Output: Công nhân được trả lương bằng thóc, một người bình thường thu được 5 bao rưỡi thóc mỗi tháng, trong khi người quản đốc được hưởng 7 bao nhiêu?


In [ ]:
import boto3
import json

# Name of the deployed SageMaker endpoint
endpoint_name = "huggingface-pytorch-inference-2025-08-02-05-20-25-896"

# Create a runtime client for SageMaker
runtime = boto3.client('sagemaker-runtime')

# Example input text
payload = {
    "inputs": """Cục Quản lý Dược, Bộ Y tế vừa ra quyết định đình chỉ lưu hành, thu hồi và yêu cầu tiêu hủy toàn quốc đối với lô sản phẩm sữa rửa mặt
      Gammaphil - chai 125ml do phát hiện chứa các chất không nằm trong công thức đã được công bố.""",
    "parameters": {"max_length": 128}
}

# Invoke the endpoint
response = runtime.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",
    Body=json.dumps(payload)
)

# Parse the response
result = json.loads(response['Body'].read().decode())
print("Model output:", result)


C:\Users\Hiep\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cục Quản lý Dược, Bộ Y tế vừa ra lệnh cấm mẹ nó lưu hành, thu hồi với yêu cầu tiêu hủy hết mẹ cái lô sữa rửa mặt Gammaphil - chai 125ml vì tội không đúng công thức.


In [3]:
### REAL

from sagemaker.huggingface import HuggingFaceModel
import sagemaker

role = "arn:aws:iam::014498663963:role/service-role/AmazonSageMaker-ExecutionRole-20250504T220510"
sess = sagemaker.Session()

huggingface_model = HuggingFaceModel(
    model_data="s3://hiep-delta-bk/models/sg/vit5-finetune-2025-07-31-15-24-27-370/output/model.tar.gz",
    role=role,
    transformers_version="4.49.0", 
    pytorch_version="2.6.0",
    py_version="py312",
    env={
        "HF_TASK": "text2text-generation" 
    }
)


predictor = huggingface_model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.xlarge",  
)


------!

In [15]:
from transformers import MT5Tokenizer, MT5ForConditionalGeneration

model_name = 'ntphiep/viT5_tst_chinese'

tokenizer = MT5Tokenizer.from_pretrained(model_name)
model = MT5ForConditionalGeneration.from_pretrained(model_name)

def paraphase(text):
    inputs = tokenizer(text, padding='longest', max_length=64, return_tensors='pt')
    input_ids = inputs.input_ids
    attention_mask = inputs.attention_mask
    output = model.generate(input_ids, attention_mask=attention_mask, max_length=64)
    return tokenizer.decode(output[0], skip_special_tokens=True)

texts = [
    "Thật tự hào khi là sinh viên trường Đại học Tôn Đức Thắng",
    "Trong thời kỳ thuộc địa , Malaysia hợp tác chủ yếu ở những nơi nào ?",
    "Thứ nhất, phải có cái ý tưởng nào mà nó chất hơn nước cất, hơn hẳn cái của bọn thường.",
    "Làm thế nào để học ngôn ngữ Java",
    "Ngoài ra, nắng nóng còn có thể gây tình trạng mất nước, kiệt sức, đột qụy do sốc nhiệt đối với cơ thể người khi tiếp xúc lâu với nền nhiệt độ cao."
]

for text in texts:
    print(f'Input: {text}')
    print(f'Output: {paraphase(text) })')
    print('-'*100)


Input: Thật tự hào khi là sinh viên trường Đại học Tôn Đức Thắng
Output: Thật là vinh hạnh khi được là học sinh của Tôn Đức Thắng Đại Học.)
----------------------------------------------------------------------------------------------------
Input: Trong thời kỳ thuộc địa , Malaysia hợp tác chủ yếu ở những nơi nào ?
Output: Thuở thuộc địa, Mã Lai quốc thường xuyên giao hảo, chủ yếu tại những địa phương nào?)
----------------------------------------------------------------------------------------------------
Input: Thứ nhất, phải có cái ý tưởng nào mà nó chất hơn nước cất, hơn hẳn cái của bọn thường.
Output: Thứ nhất, cần phải có diệu kế nào, chất lượng vượt trội hơn nước cất, vượt xa bọn thường nhân.)
----------------------------------------------------------------------------------------------------
Input: Làm thế nào để học ngôn ngữ Java
Output: Phương pháp học tập ngôn ngữ Java là gì?)
---------------------------------------------------------------------------------------------------